# OS Distribution 1001P

Inspect the `OS` distribution and common binary survival thresholds in `endpoints_1001Prostate.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DATA_ROOT = Path('/Users/marcalbesa/Desktop/TFM/data/1001Prostate')
ENDPOINTS_PATH = DATA_ROOT / 'endpoints_1001Prostate.csv'
assert ENDPOINTS_PATH.exists(), ENDPOINTS_PATH


In [ ]:
df = pd.read_csv(ENDPOINTS_PATH)
df.columns.tolist()


In [ ]:
OS_COL = 'OS'
STATUS_COL = 'Patient Status'

os_months = pd.to_numeric(df[OS_COL], errors='coerce')
status = df[STATUS_COL].astype(str).str.strip()

summary_df = pd.DataFrame([
    {'metric': 'n_rows', 'value': int(len(df))},
    {'metric': 'n_non_null_os', 'value': int(os_months.notna().sum())},
    {'metric': 'n_null_os', 'value': int(os_months.isna().sum())},
    {'metric': 'min_os', 'value': float(os_months.min())},
    {'metric': 'q25_os', 'value': float(os_months.quantile(0.25))},
    {'metric': 'median_os', 'value': float(os_months.median())},
    {'metric': 'mean_os', 'value': float(os_months.mean())},
    {'metric': 'q75_os', 'value': float(os_months.quantile(0.75))},
    {'metric': 'max_os', 'value': float(os_months.max())},
    {'metric': 'std_os', 'value': float(os_months.std())},
    {'metric': 'dead_count', 'value': int((status == 'Dead').sum())},
    {'metric': 'alive_count', 'value': int((status == 'Alive').sum())},
])

display(summary_df)
print(status.value_counts(dropna=False).to_string())


In [ ]:
bin_rows = []
for lo, hi in [(0, 6), (6, 12), (12, 24), (24, 36), (36, 60), (60, 120)]:
    count = int(((os_months >= lo) & (os_months < hi)).sum())
    bin_rows.append({'bin': f'[{lo}, {hi})', 'count': count})

bin_df = pd.DataFrame(bin_rows)
display(bin_df)


In [ ]:
threshold_rows = []
for thr in [6, 9, 12, 18, 21, 24, 36, 60]:
    label = (os_months >= thr).astype('Int64')
    threshold_rows.append({
        'label_name': f'os_{thr}_label',
        'threshold_months': thr,
        'positive_count': int((label == 1).sum()),
        'negative_count': int((label == 0).sum()),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(os_months.dropna(), bins=20, color='#2c7da0', edgecolor='white')
axes[0].axvline(os_months.median(), color='#9c2f2f', linestyle='--', linewidth=2, label=f'Median = {os_months.median():.2f}')
axes[0].axvline(os_months.mean(), color='#1b4332', linestyle=':', linewidth=2, label=f'Mean = {os_months.mean():.2f}')
axes[0].set_title('OS Distribution (months)')
axes[0].set_xlabel('OS')
axes[0].set_ylabel('Patients')
axes[0].legend(frameon=False)

status_counts = status.value_counts()
axes[1].bar(status_counts.index, status_counts.values, color=['#bc4749', '#4d908e'])
axes[1].set_title('Patient Status')
axes[1].set_ylabel('Patients')

plt.tight_layout()
plt.show()
